In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision matplotlib numpy diffusers transformers accelerate -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

_install('torch')
_install('torchvision')
_install('matplotlib')
_install('numpy')
_install('diffusers')
_install('transformers')
_install('accelerate')

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Diffusion Meets Flow Matching 代码实战：学习实现 vs 工程实现

基于文章 *Diffusion Meets Flow Matching: Two Sides of the Same Coin*，
用 **2D 概率路径生成 + diffusers 工程推理** 演示 Diffusion 与 Flow Matching 的统一视角。

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解噪声预测与速度场预测为何可以互相换算 | 掌握现代扩散库如何封装 scheduler / pipeline / guidance |
| 实现方式 | 2D toy distribution 上完整训练 `ε`-prediction 与 `v`-prediction | `diffusers` 做 inference-only 演示与 scheduler 替换 |
| 代码量 | 中等 | 很少 |
| 适合场景 | 面试准备、原理深入 | 工程落地、快速验证 |

> 两条路径不是两套无关的方法，而是同一条时间相关概率路径在不同抽象层级上的表达。

## 1. 论文背景与最小理论

文章的统一起点是：

$$x_t = \alpha_t x + \sigma_t \epsilon,$$

其中 `x` 是真实样本，`\epsilon` 是高斯噪声。

- Diffusion 常写成：预测噪声 `\hat\epsilon_\theta(x_t,t)`
- Flow Matching 常写成：预测速度场 `v_\theta(x_t,t)`

若对路径求导，则真实速度为：

$$u_t(x,\epsilon) = \dot{\alpha}_t x + \dot{\sigma}_t \epsilon.$$

若模型先预测噪声，也可以恢复干净样本估计：

$$\hat{x} = \frac{x_t - \sigma_t \hat\epsilon_\theta}{\alpha_t}. $$

再把 `x_t`、`\hat\epsilon` 代入，就能得到速度形式。这说明二者描述的是同一个动态过程，只是参数化不同。

## 2. 组件映射表

| 论文组件 | 学习路径覆盖 | 工程库/API 对应 | 状态 |
|---------|-------------|----------------|------|
| 概率路径 `x_t = α_t x + σ_t ε` | 完整实现 + 可视化 | scheduler 控制的 latent trajectory | Runnable |
| 噪声预测 `ε_θ(x_t,t)` | 完整实现 + 训练 | U-Net / pipeline 内部 noise prediction | Runnable |
| 干净样本恢复 `x̂` | 完整实现 | scheduler 内部更新逻辑 | Runnable |
| 真实速度 `u_t` | 完整实现 + 监督目标 | ODE / solver 视角 | Runnable |
| 速度场预测 `v_θ(x_t,t)` | 完整实现 + 训练 | solver abstraction | Runnable |
| `ε` 与 `v` 的线性换算 | 公式 + 代码验证 | 工程中通常被封装 | Runnable |
| 扩散离散采样 | 完整实现 | `scheduler.step()` | Runnable |
| Flow Matching ODE 积分 | 完整实现（Euler） | ODE-style solver | Runnable |
| CFG / inference steps / scheduler swap | 理论说明 | `guidance_scale` / `num_inference_steps` / `from_config` | Illustrative |

## 3. 数据准备

为了让数学关系清楚、训练稳定且 CPU 友好，这里使用 **2D 双月形数据**。

原因：
- 生成目标是连续分布，适合演示从噪声回到数据分布
- 训练只需小 MLP，不依赖大模型
- 可以直接把真实分布、生成分布和采样轨迹画出来

In [ ]:
def generate_two_moons(n_samples=2048, noise_std=0.06):
    n1 = n_samples // 2
    n2 = n_samples - n1

    theta1 = torch.linspace(0, math.pi, n1)
    x1 = torch.cos(theta1)
    y1 = torch.sin(theta1)

    theta2 = torch.linspace(0, math.pi, n2)
    x2 = 1.0 - torch.cos(theta2)
    y2 = 1.0 - torch.sin(theta2) - 0.5

    x = torch.cat([x1, x2])
    y = torch.cat([y1, y2])
    data = torch.stack([x, y], dim=1)
    data = data + noise_std * torch.randn_like(data)
    data = (data - data.mean(0)) / data.std(0)
    return data

points = generate_two_moons()
train_loader = DataLoader(TensorDataset(points), batch_size=256, shuffle=True)
print('Dataset shape:', points.shape)
print('Number of batches:', len(train_loader))

plt.figure(figsize=(5, 5))
plt.scatter(points[:, 0].numpy(), points[:, 1].numpy(), s=4, alpha=0.5)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Two-moons training distribution')
plt.axis('equal')
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# ── 超参数（两条路径共用，集中管理） ──
DATA_DIM = 2
TIME_DIM = 32
HIDDEN_DIM = 128
NUM_EPOCHS = 120
LR = 1e-3
NUM_STEPS = 80


def alpha_sigma(t):
    alpha = torch.cos(0.5 * math.pi * t)
    sigma = torch.sin(0.5 * math.pi * t)
    return alpha, sigma


def alpha_sigma_dot(t):
    alpha_dot = -0.5 * math.pi * torch.sin(0.5 * math.pi * t)
    sigma_dot = 0.5 * math.pi * torch.cos(0.5 * math.pi * t)
    return alpha_dot, sigma_dot


def sample_path(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    alpha, sigma = alpha_sigma(t)
    x_t = alpha.unsqueeze(1) * x0 + sigma.unsqueeze(1) * noise
    return x_t, noise


def true_velocity(x0, noise, t):
    alpha_dot, sigma_dot = alpha_sigma_dot(t)
    return alpha_dot.unsqueeze(1) * x0 + sigma_dot.unsqueeze(1) * noise


def epsilon_to_velocity(x_t, eps_hat, t):
    alpha, sigma = alpha_sigma(t)
    alpha_dot, sigma_dot = alpha_sigma_dot(t)
    a = alpha_dot / alpha.clamp_min(1e-5)
    b = sigma_dot - a * sigma
    return a.unsqueeze(1) * x_t + b.unsqueeze(1) * eps_hat


def predict_x0_from_epsilon(x_t, eps_hat, t):
    alpha, sigma = alpha_sigma(t)
    return (x_t - sigma.unsqueeze(1) * eps_hat) / alpha.unsqueeze(1).clamp_min(1e-5)


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.linspace(0, -math.log(10000), half, device=t.device))
        args = t.unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class TinyVectorField(nn.Module):
    def __init__(self, data_dim=2, time_dim=32, hidden_dim=128):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(time_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x_t, t):
        t_emb = self.time_emb(t)
        return self.net(torch.cat([x_t, t_emb], dim=1))


def train_model(model, loader, loss_mode='epsilon', num_epochs=NUM_EPOCHS, lr=LR):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for (x0,) in loader:
            x0 = x0.to(device)
            t = torch.rand(x0.shape[0], device=device).clamp(1e-4, 1 - 1e-4)
            x_t, noise = sample_path(x0, t)
            pred = model(x_t, t)
            target = noise if loss_mode == 'epsilon' else true_velocity(x0, noise, t)
            loss = nn.functional.mse_loss(pred, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        history.append(total_loss / n_batches)
        if (epoch + 1) % 20 == 0:
            print(f'{loss_mode:8s} | epoch {epoch+1:3d}/{num_epochs} | loss {history[-1]:.6f}')

    return model, history


def plot_loss_curves(loss_eps, loss_vel):
    plt.figure(figsize=(7, 4))
    plt.plot(loss_eps, label='epsilon prediction')
    plt.plot(loss_vel, label='velocity prediction')
    plt.xlabel('Epoch')
    plt.ylabel('MSE loss')
    plt.title('Training loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 4. 学习路径：`ε`-prediction

扩散视角中，模型学习：

$$\hat\epsilon_\theta(x_t, t) \approx \epsilon.$$

训练目标：

$$\mathcal{L}_{DM} = \mathbb{E}_{x,\epsilon,t} [\|\hat\epsilon_\theta(x_t,t)-\epsilon\|^2].$$

In [ ]:
eps_model = TinyVectorField(DATA_DIM, TIME_DIM, HIDDEN_DIM)
eps_model, eps_loss_history = train_model(eps_model, train_loader, loss_mode='epsilon')
print('epsilon model ready')

### 从 `ε̂` 恢复 `x̂` 与速度

若已经有 `ε̂`，就能恢复干净样本估计：

$$\hat{x} = \frac{x_t - \sigma_t \hat\epsilon}{\alpha_t}. $$

进一步可换算出速度参数化：

$$\hat{v}(x_t,t) = \frac{\dot{\alpha}_t}{\alpha_t}x_t + \left(\dot{\sigma}_t - \frac{\dot{\alpha}_t}{\alpha_t}\sigma_t\right) \hat\epsilon.$$

In [ ]:
batch = points[:512].to(device)
t_demo = torch.full((batch.shape[0],), 0.35, device=device)
x_t_demo, noise_demo = sample_path(batch, t_demo)

with torch.no_grad():
    eps_hat_demo = eps_model(x_t_demo, t_demo)
    x0_hat_demo = predict_x0_from_epsilon(x_t_demo, eps_hat_demo, t_demo)
    v_from_eps_demo = epsilon_to_velocity(x_t_demo, eps_hat_demo, t_demo)
    v_true_demo = true_velocity(batch, noise_demo, t_demo)

print('x_t shape:', tuple(x_t_demo.shape))
print('eps_hat shape:', tuple(eps_hat_demo.shape))
print('x0_hat shape:', tuple(x0_hat_demo.shape))
print('velocity-from-eps shape:', tuple(v_from_eps_demo.shape))
print('velocity conversion MSE:', nn.functional.mse_loss(v_from_eps_demo, v_true_demo).item())

## 5. 学习路径：`v`-prediction

Flow Matching 直接拟合真实速度场：

$$u_t(x,\epsilon) = \dot{\alpha}_t x + \dot{\sigma}_t \epsilon.$$

训练目标：

$$\mathcal{L}_{FM} = \mathbb{E}_{x,\epsilon,t} [\|v_\theta(x_t,t)-u_t(x,\epsilon)\|^2].$$

In [ ]:
vel_model = TinyVectorField(DATA_DIM, TIME_DIM, HIDDEN_DIM)
vel_model, vel_loss_history = train_model(vel_model, train_loader, loss_mode='velocity')
print('velocity model ready')
plot_loss_curves(eps_loss_history, vel_loss_history)

## 6. 训练 vs 推理的区别

| 阶段 | 学习路径行为 | 工程路径行为 |
|------|------------|------------|
| 训练 | 在随机 `t` 上采样 `x_t`，拟合 `ε` 或 `u_t` | 本 Notebook 不训练工程模型，只调用预训练 pipeline |
| 推理 | 从纯噪声出发，按离散更新或 Euler 积分回到数据分布 | `pipeline(prompt, num_inference_steps=...)` + scheduler 执行多步去噪 |

In [ ]:
@torch.no_grad()
def sample_with_epsilon_model(model, n_samples=2048, num_steps=NUM_STEPS):
    model.eval()
    x = torch.randn(n_samples, DATA_DIM, device=device)
    traj = [x.detach().cpu()]
    ts = torch.linspace(1.0, 0.0, num_steps + 1, device=device)

    for i in range(num_steps):
        t_cur = torch.full((n_samples,), ts[i], device=device)
        t_next = torch.full((n_samples,), ts[i + 1], device=device)
        eps_hat = model(x, t_cur)
        x0_hat = predict_x0_from_epsilon(x, eps_hat, t_cur)
        alpha_next, sigma_next = alpha_sigma(t_next)
        x = alpha_next.unsqueeze(1) * x0_hat + sigma_next.unsqueeze(1) * eps_hat
        if i in {0, num_steps // 3, 2 * num_steps // 3, num_steps - 1}:
            traj.append(x.detach().cpu())
    return x.detach().cpu(), traj


@torch.no_grad()
def sample_with_velocity_model(model, n_samples=2048, num_steps=NUM_STEPS):
    model.eval()
    x = torch.randn(n_samples, DATA_DIM, device=device)
    traj = [x.detach().cpu()]
    ts = torch.linspace(1.0, 0.0, num_steps + 1, device=device)

    for i in range(num_steps):
        t_cur = torch.full((n_samples,), ts[i], device=device)
        dt = ts[i] - ts[i + 1]
        v = model(x, t_cur)
        x = x - dt * v
        if i in {0, num_steps // 3, 2 * num_steps // 3, num_steps - 1}:
            traj.append(x.detach().cpu())
    return x.detach().cpu(), traj


eps_samples, eps_traj = sample_with_epsilon_model(eps_model)
vel_samples, vel_traj = sample_with_velocity_model(vel_model)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(points[:, 0].numpy(), points[:, 1].numpy(), s=4, alpha=0.35)
axes[0].set_title('Real distribution')
axes[1].scatter(eps_samples[:, 0].numpy(), eps_samples[:, 1].numpy(), s=4, alpha=0.35)
axes[1].set_title('Generated by epsilon model')
axes[2].scatter(vel_samples[:, 0].numpy(), vel_samples[:, 1].numpy(), s=4, alpha=0.35)
axes[2].set_title('Generated by velocity model')
for ax in axes:
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.axis('equal')
    ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, len(eps_traj), figsize=(3 * len(eps_traj), 6))
for i, step_points in enumerate(eps_traj):
    axes[0, i].scatter(step_points[:, 0], step_points[:, 1], s=3, alpha=0.3)
    axes[0, i].set_title(f'epsilon step {i}')
    axes[0, i].set_xlabel('x1')
    axes[0, i].set_ylabel('x2')
    axes[0, i].axis('equal')

for i, step_points in enumerate(vel_traj):
    axes[1, i].scatter(step_points[:, 0], step_points[:, 1], s=3, alpha=0.3)
    axes[1, i].set_title(f'velocity step {i}')
    axes[1, i].set_xlabel('x1')
    axes[1, i].set_ylabel('x2')
    axes[1, i].axis('equal')

plt.tight_layout()
plt.show()

## 7. 工程路径：用 diffusers 观察工业抽象

在真实工程中，我们通常不会自己手写完整采样循环，而是直接用高层 pipeline：

- `DiffusionPipeline.from_pretrained(...)`：加载模型与组件
- `.to(device)`：放到 CPU / GPU
- `pipeline(prompt=..., num_inference_steps=..., guidance_scale=...)`：执行推理
- `Scheduler.from_config(pipe.scheduler.config)`：替换 sampler

这里优先演示轻量 tiny pipeline。实际项目里通常会换成 `runwayml/stable-diffusion-v1-5` 之类模型。

In [ ]:
ENG_READY = False
try:
    from diffusers import DiffusionPipeline, DDIMScheduler, EulerDiscreteScheduler

    ENGINEERING_MODEL_ID = 'hf-internal-testing/tiny-stable-diffusion-pipe'
    REAL_WORLD_MODEL_ID = 'runwayml/stable-diffusion-v1-5'
    dtype = torch.float16 if device.type == 'cuda' else torch.float32

    pipe = DiffusionPipeline.from_pretrained(ENGINEERING_MODEL_ID, torch_dtype=dtype)
    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    prompt = 'a tiny moon over mountains, watercolor style'
    out_default = pipe(prompt=prompt, num_inference_steps=10, guidance_scale=5.0)
    img_default = out_default.images[0]

    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    out_ddim = pipe(prompt=prompt, num_inference_steps=10, guidance_scale=5.0)
    img_ddim = out_ddim.images[0]

    pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
    out_euler = pipe(prompt=prompt, num_inference_steps=10, guidance_scale=5.0)
    img_euler = out_euler.images[0]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_default)
    axes[0].set_title('Default scheduler')
    axes[1].imshow(img_ddim)
    axes[1].set_title('DDIM scheduler')
    axes[2].imshow(img_euler)
    axes[2].set_title('Euler scheduler')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print('Engineering demo model:', ENGINEERING_MODEL_ID)
    print('Real-world model example:', REAL_WORLD_MODEL_ID)
    print('Pipeline call returns:', type(out_default).__name__)
    ENG_READY = True
except Exception as e:
    print('Engineering demo skipped:', e)
    print('If needed, run this later on network-enabled Colab with a GPU.')

## 8. 学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---------|---------|---------|
| 教育价值 | 直接理解 `ε`、`x̂`、`v`、ODE、scheduler 的数学关系 | 理解工业级 API 如何组织推理流程 |
| 代码量 | 中等，需要自己写训练与采样 | 很少，主要关心输入参数与输出 |
| 灵活性 | 高，可以改路径、损失、更新规则 | 中等，受限于库接口与已有组件 |
| 稳定性 | 适合教学，但需要自己处理细节 | 高，依赖成熟实现与社区验证 |
| 工业适配度 | 更适合研究原型或讲解 | 更适合产品验证与部署链路 |
| 适用场景 | 面试准备、理论理解、算法实验 | 快速试模型、改 scheduler、做 prompt inference |

## 9. 结果与边界

- 在同一条路径 `x_t = α_t x + σ_t ε` 下，`ε`-prediction 与 `v`-prediction 可以互相映射
- 2D toy data 已足够展示两种训练目标如何把噪声端推回数据端
- 工程世界中的 `pipeline + scheduler`，本质上就是把这种多步更新过程封装成更高层接口

边界：
- 学习路径证明的是**参数化统一性**，不等于复现完整图像扩散大模型
- 工程路径展示的是**高层调用方式**，不等于解释了 U-Net、VAE、text encoder 的全部内部实现

## 10. 面试与项目表达

### 高频面试题

**Q1: Diffusion 和 Flow Matching 的本质区别是什么？**
- Diffusion 常把学习目标写成噪声预测 `ε_θ(x_t, t)`
- Flow Matching 常把学习目标写成速度场预测 `v_θ(x_t, t)`
- 若二者基于同一条概率路径，它们不是两种独立方法，而是同一动态过程的不同参数化

**Q2: 为什么说二者是 Two Sides of the Same Coin？**
- 因为 `x_t = α_t x + σ_t ε` 给了统一路径
- 对路径求导可得到真实速度 `u_t`
- 知道 `ε` 可以恢复速度，知道速度也能映射回扩散视角

**Q3: scheduler 在工程里到底是什么？**
- 它控制时间步序列与每一步状态更新规则
- 从数学角度看，scheduler 就是在做离散积分 / 数值求解

**Q4: guidance_scale 的工程意义是什么？**
- 它控制条件信息对生成方向的影响强度
- 值更大通常意味着更贴 prompt，但也可能牺牲多样性

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|------|-----------|
| 1 | Diffusion 和 Flow Matching 的关系？ | 同一路径上的两种参数化：一个预测噪声，一个预测速度 |
| 2 | 为什么能从 `ε̂` 恢复速度？ | 因为真实速度可写成 `x_t` 与 `ε` 的线性组合 |
| 3 | scheduler 的本质是什么？ | 多步去噪过程的离散积分器 |
| 4 | 为什么 toy 2D demo 有意义？ | 它把抽象公式变成可视化轨迹，最适合讲原理 |
| 5 | guidance_scale 调的是什么？ | prompt 对齐度与生成多样性的平衡 |

### 项目表达

- **学习深度**：我从同一条概率路径出发，分别实现了 `ε`-prediction 和 `v`-prediction，验证了 Diffusion 与 Flow Matching 的统一性
- **工程能力**：我用 `diffusers` 跑通了预训练 pipeline，并演示了 scheduler 替换与 inference 参数调节
- **对比思考**：我能讲清楚手写最小实现与工业库抽象之间的关系

### 参考资料
- https://docs.pytorch.org/vision/main/datasets.html
- https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html
- https://huggingface.co/docs/diffusers/index
- https://huggingface.co/docs/hub/diffusers